# DiffusionRefined — decay of a magnetic Fourier mode (refined mesh)

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``diffusion_refined``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

Same physics as `diffusion` (Fourier-mode decay, $\eta=0.05$, analytic
$e^{-\eta\pi^2t}$) on a finer mesh: the L2 error against the analytic solution
must decrease under refinement. Taken together with the coarse `diffusion`
notebook this establishes mesh convergence; the tolerance here is tighter than
on the coarse mesh.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 2 | $\eta$ used verbatim as diffusion coefficient | decay rate |
| — | mesh convergence of the discretised diffusion operator | tighter error on the refined mesh |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "diffusion_refined.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition diffusion_refined.data 2 -> PAR_diffusion_refined.data + DOM_000*.Zones
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
ETA, K = 0.05, np.pi
def analytic_ratio(t): return np.exp(-ETA * K**2 * np.asarray(t, float))

TAGS = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
err_refine = ratio_error(case.dataFileName_, TAGS, analytic_ratio)
print(f"refined max |ratio - analytic| : {err_refine:.4e}")
# diffusion.ipynb asserts err_coarse < 0.05 on the coarse mesh; the refined mesh
# must do strictly better (mesh convergence).
assert err_refine < 0.02, f"refined decay error too large: {err_refine:.3e}"
print("OK: refined mesh matches the analytic solution within the tighter tolerance.")

import matplotlib.pyplot as plt
t, b = read_son(case.dataFileName_, "SONDE_Y050")
plt.figure(figsize=(6,4))
plt.plot(t, b/b[0], "o-", ms=2, label="TRUST-LiMe (refined)")
plt.plot(t, analytic_ratio(t), "k--", label="analytic exp(-eta pi^2 t)")
plt.xlabel("t [s]"); plt.ylabel("b(t)/b(0)"); plt.legend(); plt.grid(True); plt.show()


## Computer Performance


In [ ]:
run.tablePerf()